In [1]:
import json
import nltk
import string
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from nltk.stem.wordnet import WordNetLemmatizer

In [2]:
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
lm = WordNetLemmatizer()

ourClasses = []
newWords = []
documentX = []
documentY = []

f = open('/content/FAQs_8.json')
data = json.load(f)

for i in data['intents']:
    for pattern in i["patterns"]:
        ournewTKns = nltk.word_tokenize(pattern)
        newWords.extend(ournewTKns)
        documentX.append(pattern)
        documentY.append(i["tag"])

    if i ["tag"] not in ourClasses:
        ourClasses.append(i["tag"])

newWords = [lm.lemmatize(word.lower()) for word in newWords if word not in string.punctuation]
newWords = sorted(set(newWords))
ourClasses = sorted(set(ourClasses))

In [4]:
len(newWords)

345

In [5]:
tfidf_vectorizer = TfidfVectorizer()

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Transform the entire dataset using TF-IDF vectorizer
X_tfidf = tfidf_vectorizer.fit_transform(documentX).toarray()

# Create and train Random Forest classifier on the entire dataset
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_tfidf, documentY)

# Make predictions on the same dataset
y_pred_tfidf = rf_classifier.predict(X_tfidf)


In [7]:
# Evaluate the model
accuracy = accuracy_score(documentY, y_pred_tfidf)
print("Accuracy:", accuracy)

Accuracy: 0.9972826086956522


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

precision = precision_score(documentY, y_pred_tfidf, average='weighted')
recall = recall_score(documentY, y_pred_tfidf, average='weighted')
f1 = f1_score(documentY, y_pred_tfidf, average='weighted')

# Print the evaluation metrics
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.9977355072463768
Recall: 0.9972826086956522
F1 Score: 0.9973346161847306


In [9]:
#X_tfidf = tfidf_vectorizer.fit_transform(documentX).toarray()

# Split the data into training and testing sets
#X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, documentY, test_size=0.2, random_state=42)

# Create and train Random Forest classifier
#rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
#rf_classifier.fit(X_train_tfidf, y_train_tfidf)

In [10]:
#from sklearn.metrics import accuracy_score, precision_score, recall_score

# Make predictions on the test set
#y_pred_tfidf = rf_classifier.predict(X_test_tfidf)

# Evaluate the model
#accuracy = accuracy_score(y_test_tfidf, y_pred_tfidf)
#precision = precision_score(y_test_tfidf, y_pred_tfidf, average='weighted', zero_division=1)
#recall = recall_score(y_test_tfidf, y_pred_tfidf, average='weighted', zero_division=1)

# Print the metrics
#print("Accuracy:", accuracy)
#print("Precision:", precision)
#print("Recall:", recall)

In [11]:
def remove_stopwords(text):
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    words = nltk.word_tokenize(text)
    filtered_text = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_text)

def Pclass(text, vectorizer, classifier, labels):
    text_without_stopwords = remove_stopwords(text)
    text_tfidf = vectorizer.transform([text_without_stopwords]).toarray()
    predicted_label = classifier.predict(text_tfidf)[0]
    return [predicted_label]

def getRes(intents, json_data):
    if "admission_process" in intents:
        return handleAdmissionProcess(intents, json_data)
    #elif "program inquiry" in intents:
        #return handleProgramInquiry(intents, json_data)
    elif len(intents) == 0:
        tag = 'noanswer'
    else:
        tag = intents[0]

    list_of_intents = json_data["intents"]

    for i in list_of_intents:
        if i["tag"] == tag:
            our_result = random.choice(i["responses"])
            break
    return our_result

#def handleProgramInquiry(intents, json_data):
    #if "graduate" in intents:
        #response_tag = "graduate_programs"
   # elif "undergraduate" in intents:
        #response_tag = "undergraduate_programs"

def handleAdmissionProcess(intents, json_data):
    # Check if the user specified whether they are a continuing or new student
    if "continuing" in intents:
        response_tag = "admission_process_continuing"
    elif "new" in intents:
        response_tag = "admission_process_new"
    else:
        response_tag = "admission_process_default"

    # Get the response for the specified admission process scenario
    list_of_intents = json_data["intents"]
    for i in list_of_intents:
        if i["tag"] == response_tag:
            our_result = random.choice(i["responses"])
            break

    return our_result

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import difflib

threshold_similarity = 0.5


In [13]:
threshold_close_match = 0.8  # Adjust this threshold for close matches

# ...
print("Hello! I'm your chatbot assistant. I answer frequently asked questions for Admission and Scholarship Office.")
print("Here are some of the FAQs you can ask:")
print("- Tell me about the admission process.")
print("- Are there any scholarship offers?")
print("- Can you provide information about the courses offered?")

while True:
    user_input = input("User: ")
    user_input = remove_stopwords(user_input)
    print(f"User: {user_input}")
    new_message = user_input.lower()
    intents = Pclass(new_message, tfidf_vectorizer, rf_classifier, ourClasses)

    # Get cosine similarities
    similarities = cosine_similarity(tfidf_vectorizer.transform([new_message]).toarray(), X_tfidf)
    # Find the closest match
    closest = np.argmax(similarities)
    print(similarities[0, closest])

    # Check if the closest match is below the similarity threshold
    if similarities[0, closest] < threshold_similarity:
        # Use difflib to get close matches
        close_matches = difflib.get_close_matches(new_message, newWords, n=1, cutoff=threshold_close_match)
        if close_matches:
            print(f"Did you mean: {close_matches[0]}?")
        else:
            print("Sorry, I couldn't understand your question. Please ask relevant questions or rephase it")
    else:
        our_result = getRes(intents, data)
        print("ChatBot:", our_result)

    if "goodbye" in intents:
        break

Hello! I'm your chatbot assistant. I answer frequently asked questions for Admission and Scholarship Office.
Here are some of the FAQs you can ask:
- Tell me about the admission process.
- Are there any scholarship offers?
- Can you provide information about the courses offered?
User: bye
User: bye
1.0
ChatBot: Bye! Come back again soon.
